# Landmark Chronos Feature Extraction

Generate Chronos embedding features over the landmark feature table (`general_db.csv`, produced by `new_analysis.ipynb`) and persist them as `general_db_lm_chronos.csv` (consumed by `landmarks/scripts/grid_search.py`). Each (label, subject, rep) sequence is reduced to a single feature vector by averaging Chronos encoder outputs across time and feature axes.

## Configuration

In [ ]:
INPUT_CSV = "general_db.csv"
OUTPUT_CSV = "general_db_lm_chronos.csv"
CHRONOS_MODEL = "amazon/chronos-t5-small"


## Setup

In [ ]:
import pandas as pd
import numpy as np
import os, glob
from tqdm import tqdm


## Load Chronos pipeline and source data

In [ ]:
import pandas as pd
import torch
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    CHRONOS_MODEL,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)

df = pd.read_csv(INPUT_CSV)
print(df.shape)


## Sanity-check one (label, subject, rep) slice

In [ ]:
df[(df.label == 0) & (df.subject == 0) & (df.rep == 0)]

## Compute embeddings per (label, subject, rep)

In [ ]:
db = []
subjects = []
labels = []
rep = []

for unique_label in tqdm(df["label"].unique()):
    label_data = df[df["label"] == unique_label]
    for unique_subject in label_data["subject"].unique():
        
        subject_data = label_data[label_data["subject"] == unique_subject]
        for unique_rep in subject_data["rep"].unique():
            raw_data = subject_data.drop(columns=["label", "subject", "rep"]).to_numpy()
            embeddings, state = pipeline.embed(torch.tensor(raw_data))

            features = embeddings.detach().to(torch.float32).numpy().mean(axis=0).mean(axis=0)
            db.append(features)
            subjects.append(unique_subject)
            labels.append(unique_label)
            rep.append(unique_rep)
            

## Aggregate into a DataFrame

In [ ]:
db_np = np.array(db)

In [ ]:
db_np.shape

In [ ]:
agg = {"subject": subjects, "label": labels, "rep":rep,**{ f"features_{i}":j for i,j in enumerate(np.array(db).T)}}

In [ ]:
new_db = pd.DataFrame(agg)

In [ ]:
new_db

## Save and inspect

In [ ]:
new_db.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

In [ ]:
new_db["label"].value_counts()*0.8